In [2]:
# ==========================================================
# NOTEBOOK 2
# E-COMMERCE LLM BOT
# PHI-3 MINI QLORA FINE-TUNING
#
# INPUT:
# NOTEBOOK 2
#
# OUTPUT:
# trained model
#
# ==========================================================

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -U "transformers==5.9.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 127.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


In [5]:
# !pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q peft
!pip install -q trl
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00


In [6]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "RetailBot_AI_Powered_ECommerce_Customer_Service_LLM"
)

# PROCESSED_PATH = (
#     f"{PROJECT_PATH}/data/processed"
# )

PROCESSED_PATH = (
    f"{PROJECT_PATH}/data/processed2"
)

TRAIN_PATH = (
    f"{PROCESSED_PATH}/train"
)

VAL_PATH = (
    f"{PROCESSED_PATH}/val"
)

MODEL_OUTPUT = (
    f"{PROJECT_PATH}/model/retailbot_tinyllama/processed2"
)

In [7]:
from datasets import load_from_disk

train_dataset = load_from_disk(
    TRAIN_PATH
)

validation_dataset = load_from_disk(
    VAL_PATH
)

print("Train Records:", len(train_dataset))
print("Validation Records:", len(validation_dataset))

Train Records: 27495
Validation Records: 3056


In [8]:
def formatting_function(example):

    return {
        "text":
        f"""<|user|>
{example['input']}

<|assistant|>
{example['output']}"""
    }

dataset = train_dataset.map(
    formatting_function
)

validation_dataset = validation_dataset.map(
    formatting_function
)

Map:   0%|          | 0/27495 [00:00<?, ? examples/s]

Map:   0%|          | 0/3056 [00:00<?, ? examples/s]

In [9]:
print(dataset[4]["text"])

<|user|>
I want to speak with customer service, helpme

<|assistant|>
Honored to assist! I'm clued in that you would like to speak with our customer service team. Our dedicated team is available to assist you during {{Customer Support Hours}}. Please let me know how I can help you further.


In [10]:
from transformers import AutoTokenizer

# MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
# MODEL_NAME = "microsoft/Phi-3-mini-128k-instruct"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [11]:
import transformers
print(transformers.__version__)

5.9.0


In [12]:
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [16]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # target_modules=[
    #     "q_proj",
    #     "k_proj",
    #     "v_proj",
    #     "o_proj"
    # ],
    target_modules=[
        "q_proj",
        "v_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [17]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    report_to="none"
)

In [18]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    eval_dataset=validation_dataset,
    peft_config=lora_config,
    args=training_args
)


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)


Adding EOS to train dataset:   0%|          | 0/27495 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/27495 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3056 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3056 [00:00<?, ? examples/s]

In [ ]:
# 30–60 minutes
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,1.365429
100,1.116542
150,1.038037
200,0.983240
250,0.959830
300,0.898344
350,0.875373
400,0.855131
450,0.875482
500,0.842343


TrainOutput(global_step=3437, training_loss=0.7607817701173792, metrics={'train_runtime': 2828.2219, 'train_samples_per_second': 9.722, 'train_steps_per_second': 1.215, 'total_flos': 3.337911956968243e+16, 'train_loss': 0.7607817701173792, 'entropy': 0.7280058989653716, 'mean_token_accuracy': 0.7977492185057821, 'num_tokens': 4169003.0, 'epoch': 1.0})

In [ ]:
trainer.model.save_pretrained(
    MODEL_OUTPUT
)

tokenizer.save_pretrained(
    MODEL_OUTPUT
)

print("Model Saved")
print(MODEL_OUTPUT)

Model Saved
/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama/processed2


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
import os

MODEL_OUTPUT = "/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama/processed2"

print("Exists:", os.path.exists(MODEL_OUTPUT))

for f in os.listdir(MODEL_OUTPUT):
    print(f)

Exists: True
README.md
adapter_model.safetensors
adapter_config.json
chat_template.jinja
tokenizer_config.json
tokenizer.json


In [ ]:
# import transformers
# import peft
# import trl
# import torch

# print(transformers.__version__)
# print(peft.__version__)
# print(trl.__version__)
# print(torch.__version__)

In [ ]:
# import torch

# print("GPU:", torch.cuda.get_device_name(0))
# print(
#     "Allocated:",
#     round(torch.cuda.memory_allocated()/1024**3,2),
#     "GB"
# )

In [ ]:
# print("Model Device:", next(model.parameters()).device)

# import torch
# print("CUDA:", torch.cuda.is_available())

# !nvidia-smi

In [ ]:
# import torch

# print("Torch:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))